# 🚗 City-ANPR: Train Vehicle License Plate Detector (YOLOv8s)

Welcome to the **City-ANPR Plate Detector** training notebook on Google Colab! This notebook trains a high-speed, deep-learning object detector specifically optimized to locate vehicle license plates in diverse traffic conditions.

### 🎯 What this notebook does:
1. **GPU Check & Setup**: Verifies GPU acceleration and installs Ultralytics YOLO.
2. **Dataset Loading**: Connects to Google Drive, extracts `processed_for_colab.zip`, and configures `data.yaml`.
3. **Sanity Check**: Verifies image and label counts across splits and displays sample training plates.
4. **Training**: Trains a pretrained `yolov8s.pt` with early stopping, saving checkpoints directly to Google Drive.
5. **Evaluation**: Evaluates the best checkpoint on the held-out **TEST** split (Precision, Recall, mAP50, mAP50-95).
6. **Visual Inspection**: Renders 30 test predictions and isolates failure cases (missed plates and false positives).
7. **Speed Benchmark**: Benchmarks inference latency (in milliseconds) and throughput (FPS) on the GPU.
8. **Export & Summary**: Saves evaluation plots, `results.csv`, `metrics.json`, and outputs a markdown summary.

> 💡 **Beginner Tip**: You can run each cell one by one by pressing **Shift + Enter** or clicking the Play button on the left of each cell.

## Step 1: Check GPU & Install Dependencies

Deep learning models require a GPU (Graphics Processing Unit) to train in minutes rather than hours. In this step:
- We run `!nvidia-smi` to inspect the available GPU hardware.
- If no GPU is found, a clear warning is printed guiding you to switch your Colab runtime to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).
- We install the `ultralytics` package which provides the state-of-the-art YOLOv8 implementation.

In [ ]:
# Step 1: Check GPU hardware acceleration and install Ultralytics
import os
import sys

print("=" * 65)
print("Checking GPU Hardware Acceleration in Google Colab...")
print("=" * 65)

# Run nvidia-smi command to inspect GPU details
!nvidia-smi

import torch

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"\n[OK] GPU Detected: {device_name} with {total_mem:.2f} GB VRAM")
    print("Ready to train YOLO on GPU!")
else:
    print("\n[WARNING] NO GPU DETECTED!")
    print("Training YOLO on CPU will be extremely slow.")
    print("To enable a free GPU in Google Colab:")
    print("  1. Click 'Runtime' in the top menu bar")
    print("  2. Select 'Change runtime type'")
    print("  3. Under 'Hardware accelerator', choose 'T4 GPU'")
    print("  4. Click 'Save' and re-run this notebook.")

# Install Ultralytics and dependencies
print("\nInstalling Ultralytics (YOLO) and required dependencies...")
!pip install -q ultralytics pyyaml

import ultralytics
from ultralytics import YOLO

print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"PyTorch Version:     {torch.__version__} (CUDA: {torch.version.cuda})")


## Step 2: Mount Google Drive & Extract Dataset

Colab instances are temporary virtual machines. If you disconnect, local files are deleted. By mounting **Google Drive**:
- We read your uploaded `processed_for_colab.zip` archive from `/content/drive/MyDrive/anpr/`.
- We extract the dataset to fast local Colab SSD storage at `/content/data`.
- We automatically update `data.yaml` with absolute paths so YOLO knows exactly where to find `images/train`, `images/val`, and `images/test`.

In [ ]:
# Step 2: Mount Google Drive, extract dataset, and rewrite data.yaml
import os
from pathlib import Path
import shutil
import zipfile
from google.colab import drive
import yaml

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define dataset zip and extraction paths
ZIP_PATH = Path('/content/drive/MyDrive/anpr/processed_for_colab.zip')
DATA_DIR = Path('/content/data')

print(f"Checking for dataset archive: {ZIP_PATH}")
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Dataset archive not found at {ZIP_PATH}!\n"
        f"Please ensure you created a folder named 'anpr' in your Google Drive "
        f"and uploaded 'processed_for_colab.zip' there."
    )

# Unzip dataset to /content/data if not already present
detector_candidates = [
    DATA_DIR / 'processed' / 'detector',
    DATA_DIR / 'detector'
]

if not any(p.exists() for p in detector_candidates):
    print(f"Unzipping {ZIP_PATH} to {DATA_DIR} (this takes ~15-20 seconds)...\n")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Unzip complete!")
else:
    print("Dataset already extracted in /content/data.")

# Locate detector folder
if (DATA_DIR / 'processed' / 'detector').exists():
    DETECTOR_DIR = DATA_DIR / 'processed' / 'detector'
elif (DATA_DIR / 'detector').exists():
    DETECTOR_DIR = DATA_DIR / 'detector'
else:
    raise FileNotFoundError(f"Could not locate detector folder in {DATA_DIR}")

print(f"Detector Directory: {DETECTOR_DIR}")

# Rewrite detector/data.yaml with absolute Colab paths
DATA_YAML_PATH = DETECTOR_DIR / 'data.yaml'
data_config = {
    'path': str(DETECTOR_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'plate'}
}

with open(DATA_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"\nUpdated {DATA_YAML_PATH} with absolute Colab paths:")
print(DATA_YAML_PATH.read_text())


## Step 3: Dataset Sanity Check & Visualization

Before launching a long training job, always verify your dataset:
- We check that every image has a matching `.txt` YOLO annotation file in `train`, `val`, and `test`.
- We randomly sample 6 training images and draw their ground-truth bounding boxes in lime green.
- This confirms that the coordinate system, box scaling, and plate alignments are 100% correct.

In [ ]:
# Step 3: Verify split file counts and draw sample training bounding boxes
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

print("=" * 65)
print("=== Dataset Sanity Check ===")
print("=" * 65)

splits = ['train', 'val', 'test']
print(f"{'Split':<8} | {'Images':<8} | {'Labels':<8} | {'Status':<10}")
print("-" * 42)
for s in splits:
    img_dir = DETECTOR_DIR / 'images' / s
    lbl_dir = DETECTOR_DIR / 'labels' / s
    
    n_imgs = len([f for f in img_dir.glob('*') if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    status = "OK" if (n_imgs == n_lbls and n_imgs > 0) else "MISMATCH / EMPTY"
    print(f"{s:<8} | {n_imgs:<8} | {n_lbls:<8} | {status:<10}")

# Visualize 6 random training samples with bounding boxes
train_img_dir = DETECTOR_DIR / 'images' / 'train'
train_lbl_dir = DETECTOR_DIR / 'labels' / 'train'
train_imgs = sorted([f for f in train_img_dir.glob('*') if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])

random.seed(42)
sample_imgs = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor('#0f172a')

for ax, img_path in zip(axes.flat, sample_imgs):
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
    with Image.open(img_path) as im:
        im = im.convert('RGB')
        w, h = im.size
        ax.imshow(im)
        
    if lbl_path.exists():
        lines = [l.strip().split() for l in lbl_path.read_text().splitlines() if l.strip()]
        for parts in lines:
            if len(parts) >= 5:
                cls_id = int(parts[0])
                xc, yc, bw, bh = [float(x) for x in parts[1:5]]
                # Convert normalized coordinates [0, 1] to pixel coordinates
                xmin = (xc - bw / 2.0) * w
                ymin = (yc - bh / 2.0) * h
                box_w = bw * w
                box_h = bh * h
                rect = patches.Rectangle(
                    (xmin, ymin), box_w, box_h,
                    linewidth=2.5, edgecolor='#00FF66', facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(
                    xmin, max(0, ymin - 4), "plate",
                    color='#00FF66', fontsize=10, fontweight='bold',
                    bbox=dict(facecolor='#000000', alpha=0.7, pad=1, edgecolor='none')
                )
                
    ax.set_title(f"{img_path.name[:22]}", fontsize=10, color='#38bdf8', pad=4)
    ax.axis('off')

plt.suptitle("Sample Training Images with Bounding Boxes (Sanity Check)", fontsize=16, color='#f8fafc', y=0.98)
plt.tight_layout()
plt.show()


## Step 4: Train YOLOv8s with Ultralytics

We use **`yolov8s.pt`** (YOLOv8-small), which provides an optimal balance between high detection accuracy on small objects and ultra-fast inference speed (>60 FPS).

### ⚙️ Training Parameters:
- `imgsz=640`: Standard input resolution, scaling images while maintaining aspect ratio.
- `epochs=80`: Sufficient passes for the network to converge on vehicle plate features.
- `batch=16`: Good fit for the 16 GB VRAM on Colab's T4 GPU.
- `patience=20`: Early stopping triggers if validation mAP50 does not improve for 20 epochs.
- `seed=42`: Fixed seed for reproducible weight initialization and data augmentation.
- `save_period=5`: Checkpoints saved every 5 epochs.
- `project` & `name`: Saved directly to `/content/drive/MyDrive/anpr/runs/plate_detector_yolov8s` so **progress survives unexpected disconnects**.

In [ ]:
# Step 4: Train YOLOv8s model with Drive checkpointing
import torch
from ultralytics import YOLO

# Drive directory to persist runs and checkpoints
PROJECT_DIR = Path('/content/drive/MyDrive/anpr/runs')
RUN_NAME = 'plate_detector_yolov8s'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("=== Launching YOLOv8s Training ===")
print("=" * 65)
print(f"Target checkpoints directory: {PROJECT_DIR / RUN_NAME}")
print("Config: imgsz=640, epochs=80, batch=16, patience=20, seed=42, save_period=5\n")

# Load pretrained YOLOv8-small weights
model = YOLO('yolov8s.pt')

# Execute training
train_results = model.train(
    data=str(DATA_YAML_PATH),
    imgsz=640,
    epochs=80,
    batch=16,
    patience=20,
    seed=42,
    save_period=5,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True
)

print("\n[OK] Training completed successfully!")


### 🔄 How to Resume Training After a Disconnect

If your Google Colab session disconnects (e.g. browser closed or idle timeout):
1. **Do not start over from scratch!** Your weights were saved directly to Google Drive.
2. Re-mount Google Drive (Step 2).
3. Load `last.pt` and set `resume=True`:

```python
# Run this snippet to resume interrupted training:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/anpr/runs/plate_detector_yolov8s/weights/last.pt')
model.train(resume=True)
```

Ultralytics will automatically resume from the exact epoch where it was interrupted!

## Step 5: Evaluate `best.pt` on the TEST Split

Validation data was used during training for early stopping. To measure **true real-world generalization**, we must evaluate on the completely held-out **TEST split** (`split='test'`).
- **Precision ($P$)**: When the model predicts a plate, what percentage are actual plates?
- **Recall ($R$)**: Out of all real plates on the road, what percentage did the model detect?
- **mAP@50**: Mean Average Precision at IoU threshold 0.50.
- **mAP@50-95**: Mean Average Precision across IoU thresholds from 0.50 to 0.95 (standard COCO metric).

In [ ]:
# Step 5: Evaluate best checkpoint on the held-out TEST split
import pandas as pd
from ultralytics import YOLO

BEST_WEIGHTS_PATH = PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt'
if not BEST_WEIGHTS_PATH.exists():
    alt_path = Path(f"/content/runs/detect/{RUN_NAME}/weights/best.pt")
    if alt_path.exists():
        BEST_WEIGHTS_PATH = alt_path

print("=" * 65)
print("=== Evaluating Best Checkpoint on TEST Split ===")
print("=" * 65)
print(f"Loading model weights from: {BEST_WEIGHTS_PATH}")

best_model = YOLO(str(BEST_WEIGHTS_PATH))

# Run evaluation strictly on the TEST split
test_metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split='test',
    imgsz=640,
    batch=16,
    conf=0.25,
    iou=0.60,
    device=0 if torch.cuda.is_available() else 'cpu'
)

# Extract metrics
precision = float(test_metrics.box.mp)
recall = float(test_metrics.box.mr)
map50 = float(test_metrics.box.map50)
map50_95 = float(test_metrics.box.map)
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# Format metrics into a clean table
metrics_table = pd.DataFrame([
    {"Metric": "Precision (P)", "Value": f"{precision:.4f}", "Percentage": f"{precision*100:.2f}%"},
    {"Metric": "Recall (R)", "Value": f"{recall:.4f}", "Percentage": f"{recall*100:.2f}%"},
    {"Metric": "F1-Score", "Value": f"{f1:.4f}", "Percentage": f"{f1*100:.2f}%"},
    {"Metric": "mAP@50", "Value": f"{map50:.4f}", "Percentage": f"{map50*100:.2f}%"},
    {"Metric": "mAP@50-95", "Value": f"{map50_95:.4f}", "Percentage": f"{map50_95*100:.2f}%"},
])

print("\n=== Test Split Metrics Table ===")
display(metrics_table)


## Step 6: Visualizations (Predictions & Worst Failure Analysis)

A quantitative metric (like mAP) doesn't tell the full story. In this step:
1. **Predictions Grid**: We display 30 test images with predicted bounding boxes and confidence scores.
2. **Worst Failure Analysis**: We identify the exact images where the detector made errors at confidence threshold 0.25:
   - **Missed Plates (False Negatives)**: Ground-truth plate existed, but detector missed it.
   - **Ghost Detections (False Positives)**: Model predicted a plate where none existed (e.g. headlights, grill emblems).
   - This provides actionable insight into lighting, angle, or blur limitations.

In [ ]:
# Step 6: Visualize 30 test predictions and isolate failure cases
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

test_img_dir = DETECTOR_DIR / 'images' / 'test'
test_lbl_dir = DETECTOR_DIR / 'labels' / 'test'
test_imgs = sorted([f for f in test_img_dir.glob('*') if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])

print(f"Total Test Images: {len(test_imgs)}")

# --- Part A: 30 Test Predictions ---
n_vis = min(30, len(test_imgs))
selected_test = test_imgs[:n_vis]

pred_results = best_model.predict(
    source=[str(p) for p in selected_test],
    imgsz=640,
    conf=0.25,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=False
)

fig, axes = plt.subplots(5, 6, figsize=(22, 16))
fig.patch.set_facecolor('#0f172a')

for ax, r, img_p in zip(axes.flat, pred_results, selected_test):
    im_bgr = r.plot(labels=True, conf=True, line_width=2)
    im_rgb = im_bgr[:, :, ::-1] # BGR -> RGB
    ax.imshow(im_rgb)
    n_boxes = len(r.boxes)
    ax.set_title(f"{img_p.name[:18]} ({n_boxes} plate{'s' if n_boxes != 1 else ''})", fontsize=8, color='#38bdf8', pad=3)
    ax.axis('off')

plt.suptitle("YOLOv8s Predictions on Test Split (30 Images, conf >= 0.25)", fontsize=16, color='#f8fafc', y=0.98)
plt.tight_layout()
plt.show()

# --- Part B: Worst Failure Cases Analysis ---
print("\nSearching for failure cases across all test images...")

def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    a2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

failure_cases = []

all_test_preds = best_model.predict(
    source=[str(p) for p in test_imgs],
    imgsz=640,
    conf=0.25,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=False
)

for r, img_p in zip(all_test_preds, test_imgs):
    lbl_p = test_lbl_dir / f"{img_p.stem}.txt"
    with Image.open(img_p) as im:
        img_w, img_h = im.size
    
    # Ground-truth boxes
    gt_boxes = []
    if lbl_p.exists():
        for line in lbl_p.read_text().splitlines():
            pts = line.strip().split()
            if len(pts) >= 5:
                xc, yc, bw, bh = [float(x) for x in pts[1:5]]
                gt_boxes.append([
                    (xc - bw / 2.0) * img_w,
                    (yc - bh / 2.0) * img_h,
                    (xc + bw / 2.0) * img_w,
                    (yc + bh / 2.0) * img_h,
                ])
    
    # Predicted boxes
    pred_boxes = []
    pred_confs = []
    for b in r.boxes:
        pred_boxes.append(b.xyxy[0].cpu().numpy().tolist())
        pred_confs.append(float(b.conf[0]))
    
    # Missed Plates (False Negatives)
    has_fn = False
    missed_gt = []
    for gt in gt_boxes:
        matched = any(box_iou(gt, pb) >= 0.45 for pb in pred_boxes)
        if not matched:
            has_fn = True
            missed_gt.append(gt)
            
    # Ghost Detections (False Positives)
    has_fp = False
    false_preds = []
    for pb, c in zip(pred_boxes, pred_confs):
        matched = any(box_iou(pb, gt) >= 0.45 for gt in gt_boxes)
        if not matched:
            has_fp = True
            false_preds.append((pb, c))
            
    if has_fn or has_fp:
        failure_cases.append({
            'img_path': img_p,
            'gt_boxes': gt_boxes,
            'pred_boxes': pred_boxes,
            'pred_confs': pred_confs,
            'missed_gt': missed_gt,
            'false_preds': false_preds,
            'has_fn': has_fn,
            'has_fp': has_fp
        })

print(f"Identified {len(failure_cases)} failure cases across {len(test_imgs)} test images.")

if failure_cases:
    n_fail = min(12, len(failure_cases))
    fail_cols = 4
    fail_rows = (n_fail + fail_cols - 1) // fail_cols
    fig, axes = plt.subplots(fail_rows, fail_cols, figsize=(18, 4 * fail_rows))
    fig.patch.set_facecolor('#0f172a')
    
    for idx, ax in enumerate(axes.flat):
        if idx < n_fail:
            fcase = failure_cases[idx]
            with Image.open(fcase['img_path']) as im:
                ax.imshow(im)
            
            # Ground truth in GREEN
            for gt in fcase['gt_boxes']:
                rect = patches.Rectangle(
                    (gt[0], gt[1]), gt[2] - gt[0], gt[3] - gt[1],
                    linewidth=2, edgecolor='#00FF66', facecolor='none'
                )
                ax.add_patch(rect)
                
            # Predicted boxes: RED for false positives, CYAN for correct
            for pb, c in zip(fcase['pred_boxes'], fcase['pred_confs']):
                is_fp = any(box_iou(pb, fpb[0]) > 0.99 for fpb in fcase['false_preds'])
                color = '#f87171' if is_fp else '#38bdf8'
                rect = patches.Rectangle(
                    (pb[0], pb[1]), pb[2] - pb[0], pb[3] - pb[1],
                    linewidth=2, edgecolor=color, facecolor='none', linestyle='--' if is_fp else '-'
                )
                ax.add_patch(rect)
                ax.text(pb[0], pb[1] - 3, f"{c:.2f}", color=color, fontsize=8,
                        bbox=dict(facecolor='#000000', alpha=0.7, pad=1, edgecolor='none'))
            
            # Highlight missed plates in YELLOW
            for mgt in fcase['missed_gt']:
                ax.text(mgt[0], mgt[3] + 12, "MISSED PLATE", color='#facc15', fontsize=8, fontweight='bold',
                        bbox=dict(facecolor='#000000', alpha=0.8, pad=1, edgecolor='none'))
                
            status_str = []
            if fcase['has_fn']: status_str.append("FN (Missed)")
            if fcase['has_fp']: status_str.append("FP (Ghost)")
            ax.set_title(f"{fcase['img_path'].name[:15]}: {', '.join(status_str)}", fontsize=9, color='#f87171', pad=3)
        ax.axis('off')
        
    plt.suptitle("Failure Analysis: Missed Plates (Yellow Label) & False Positives (Red Box)", fontsize=15, color='#f8fafc', y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print("Excellent! No failure cases detected on test set at confidence threshold 0.25!")


## Step 7: Measure Inference Speed (Latency Benchmark)

For real-world city traffic cameras running at 30 FPS, our detector must process each frame in **under 33.3 milliseconds** ($1000\text{ ms} / 30\text{ FPS} = 33.3\text{ ms}$).
- We warm up the GPU with 10 dummy iterations.
- We record exact CUDA execution time across test images using synchronized timing.
- We report Average, Median (P50), 95th Percentile (P95) latency, and Frames Per Second (FPS).

In [ ]:
# Step 7: Measure inference latency in ms per image and throughput (FPS)
import time
import torch

print("=" * 65)
print("=== GPU Inference Latency Benchmark ===")
print("=" * 65)

benchmark_imgs = test_imgs[:100] if len(test_imgs) >= 100 else test_imgs
print(f"Benchmarking on {len(benchmark_imgs)} images on {best_model.device}...")

# 1. Warm-up GPU
print("Warming up GPU...")
for p in benchmark_imgs[:10]:
    _ = best_model.predict(source=str(p), imgsz=640, conf=0.25, verbose=False)

# 2. Timed inference with CUDA synchronization
latencies_ms = []
prep_times = []
infer_times = []
post_times = []

print("Running timed inference passes...")
for p in benchmark_imgs:
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    res = best_model.predict(source=str(p), imgsz=640, conf=0.25, verbose=False)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.perf_counter()
    
    latencies_ms.append((t_end - t0) * 1000.0)
    speed = res[0].speed
    prep_times.append(speed.get('preprocess', 0.0))
    infer_times.append(speed.get('inference', 0.0))
    post_times.append(speed.get('postprocess', 0.0))

avg_latency = float(np.mean(latencies_ms))
median_latency = float(np.median(latencies_ms))
p95_latency = float(np.percentile(latencies_ms, 95))
fps = 1000.0 / avg_latency if avg_latency > 0 else 0.0

print(f"\n--- Latency Benchmark Results ---")
print(f"Average Latency:       {avg_latency:.2f} ms per frame")
print(f"Median (P50) Latency:  {median_latency:.2f} ms per frame")
print(f"95th-Percentile (P95): {p95_latency:.2f} ms per frame")
print(f"Throughput:            {fps:.1f} FPS")
print(f"Real-Time Capable:     {'YES (>= 30 FPS)' if fps >= 30 else 'NO (< 30 FPS)'}")
print(f"\nUltralytics Internal Breakdown:")
print(f"  - Pre-processing:    {np.mean(prep_times):.2f} ms")
print(f"  - Neural Network:    {np.mean(infer_times):.2f} ms")
print(f"  - NMS Post-process:  {np.mean(post_times):.2f} ms")


## Step 8: Save Artifacts & Markdown Summary

In this final step, we ensure all valuable artifacts are safely stored in Google Drive:
- `best.pt`: Model weights that achieved the highest mAP on validation data.
- `results.csv`: Epoch-by-epoch training and validation loss progression.
- Training curves: Precision-Recall (`PR_curve.png`), F1-confidence curve (`F1_curve.png`), and loss history (`results.png`).
- `metrics.json`: Machine-readable summary of test metrics and benchmark latencies.
- A markdown summary table is printed, ready to copy into project documentation.

In [ ]:
# Step 8: Save best weights, curves, results.csv, metrics.json, and print summary
import json
import shutil

print("=" * 65)
print("=== Saving Artifacts to Google Drive ===")
print("=" * 65)

TARGET_RUN_DIR = PROJECT_DIR / RUN_NAME
TARGET_RUN_DIR.mkdir(parents=True, exist_ok=True)

local_dir = Path(best_model.trainer.save_dir) if hasattr(best_model, 'trainer') and best_model.trainer else TARGET_RUN_DIR
print(f"Source run dir: {local_dir}")
print(f"Destination:    {TARGET_RUN_DIR}")

# Copy evaluation plots & curves to Google Drive
artifacts = [
    'results.csv',
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'F1_curve.png',
    'PR_curve.png',
    'P_curve.png',
    'R_curve.png'
]

for art in artifacts:
    src_f = local_dir / art
    dst_f = TARGET_RUN_DIR / art
    if src_f.exists() and src_f != dst_f:
        shutil.copy2(src_f, dst_f)
        print(f"  [COPIED] {art} -> Google Drive")

# Save metrics.json
metrics_dict = {
    "model_name": "plate_detector_yolov8s",
    "base_model": "yolov8s.pt",
    "imgsz": 640,
    "test_metrics": {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "map50": map50,
        "map50_95": map50_95
    },
    "latency_benchmark": {
        "avg_latency_ms": avg_latency,
        "median_latency_ms": median_latency,
        "p95_latency_ms": p95_latency,
        "throughput_fps": fps
    },
    "training_config": {
        "epochs": 80,
        "batch": 16,
        "patience": 20,
        "seed": 42,
        "save_period": 5
    }
}

metrics_json_file = TARGET_RUN_DIR / "metrics.json"
with open(metrics_json_file, 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, indent=2)
print(f"  [SAVED] metrics.json -> {metrics_json_file}")

# Generate Markdown Summary
summary_markdown = f"""
### 📊 License Plate Detector Test Evaluation Summary

- **Model Architecture**: YOLOv8-small (`yolov8s.pt`)
- **Trained On**: Indian License Plates (1,467 train images)
- **Evaluated On**: Held-out **TEST Split** ({len(test_imgs)} images, zero group leakage)
- **Input Resolution**: 640 x 640 px

#### Performance Metrics Table:
| Metric | Score | Percentage |
| :--- | :--- | :--- |
| **Precision (P)** | `{precision:.4f}` | **{precision*100:.2f}%** |
| **Recall (R)** | `{recall:.4f}` | **{recall*100:.2f}%** |
| **F1-Score** | `{f1:.4f}` | **{f1*100:.2f}%** |
| **mAP@50** | `{map50:.4f}` | **{map50*100:.2f}%** |
| **mAP@50-95** | `{map50_95:.4f}` | **{map50_95*100:.2f}%** |

#### GPU Inference Benchmark (T4 GPU):
- **Average Latency**: **{avg_latency:.2f} ms** per frame
- **95th Percentile**: **{p95_latency:.2f} ms** per frame
- **Throughput**: **{fps:.1f} FPS** (Target: $\\ge 30$ FPS for real-time video)
- **Weights Location**: `/content/drive/MyDrive/anpr/runs/plate_detector_yolov8s/weights/best.pt`
"""

print("\n" + "=" * 65)
print(summary_markdown.strip())
print("=" * 65)
